# AI² at Gravitas — Build a Research Agent over Real Filings

You are joining a small research desk. Your brief:

> **Build a research agent that can answer evidence-backed questions about company filings — and show where every important claim came from.**

For this workshop we limit the corpus to three companies — **TCS, Infosys, and HCLTech** — so downloading, parsing, and indexing stays fast enough to iterate on live in the room. Nothing in the pipeline is specific to these three; point it at a different set of official filings and it works the same way.

This is notebook **1 of 4** in the workshop series:

1. **`01_llm_foundations.ipynb`** — a plain LLM, its failure modes, temperature, structured outputs
2. **`02_evidence_layer.ipynb`** — provenance, parsing, chunking
3. **`03_indexing_and_rag.ipynb`** — vector indexing and a first RAG loop
4. **`04_hybrid_retrieval.ipynb`** — combining dense + keyword search

The slides introduce ideas; here you will immediately use them.

<details><summary><b>If you get stuck</b></summary>
Ask OpenCode to explain the surrounding code, state what you expect, or give one hint. Read the diff, and explain the change back in your own words.
</details>


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data/corpus_manifest.json').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the workshop repo root (looked for data/corpus_manifest.json in this '
        'directory and its parents). Run this notebook from inside Gravitas_Workshop_Starter.'
    )


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workshop root:', ROOT)

from config import load_workshop_env
load_workshop_env()  # load .env once, up front, so later @observe spans don't warn about missing keys

from observability.tracing import flush_langfuse, dashboard_base_url


# Part I — Start with the simplest possible AI system

## Mission 1 — What does a plain LLM actually know?

### 🧠 Concept: an LLM generates text; it is not your evidence database

A language model is very good at producing useful text from its learned parameters and the context you send in the current request. But for financial research, we care about a second question:

> **Where did this claim come from?**

A fluent answer can still be stale, approximate, or unsupported.

<img src="../assets/notebook/llm_next_token.png" width="900" alt="Workshop slide explaining next-token language modelling">


### 🧠 Two different failure modes: knowledge cutoff vs hallucination
**Knowledge cutoff / stale knowledge** means model weights do not automatically absorb a filing released after that model version was trained or updated.  
**Hallucination** means the model generates a claim that is unsupported or wrong. It can hallucinate about old facts or new facts, and even after retrieval it can misread evidence.

```text
knowledge cutoff → the model may not have the fact
hallucination    → the model may say a fact it cannot support
```
RAG helps with freshness and can reduce hallucination, but does not eliminate it.


In [ ]:
# ✍️ YOUR TURN 1 — Classify each failure mode.
failure_types = {
    'A filing was published after the model version was trained': None,  # TODO: 'knowledge cutoff' or 'hallucination'
    'The model invents a page number for a real annual report': None,  # TODO
    'The correct passage is retrieved, but the model changes 12.4% to 14.2%': None,  # TODO
}
failure_types


### 🔮 Predict before running

> **Question:** What was TCS FY26 revenue and how fast did it grow in constant currency?

Possible failure categories: **freshness / exact number / unit / period / source / citation**.


In [ ]:
# ✍️ YOUR TURN 2 — You may change this to another filing-style question.
question = 'What was TCS FY26 revenue and how fast did it grow in constant currency?'
question


📎 **Script reference:** `llm/client.py` (`call_model`, `build_messages`) — the entire plain-LLM call path lives in this one small file. Run it standalone any time with `uv run python -m llm.client`.


In [ ]:
from llm.client import call_model, build_messages

plain_answer = call_model(build_messages(question))
print(plain_answer)


In [ ]:
# 🔭 TRACE CHECKPOINT 1 — make sure the generation reached Langfuse.
flush_langfuse()
print("Langfuse:", dashboard_base_url())
print("Open Tracing → Traces and find the latest workshop-llm-call.")


### 🔭 TRACE CHECKPOINT 1 — plain LLM

Open Langfuse **now**, while the system is still simple.

Find the latest trace/generation and inspect:

1. What input did the model receive?
2. How long did the generation take?

Keep the dashboard tab open. We will return to the same execution view after adding RAG and agents.

> **Trace** = one end-to-end execution.  
> **Span / observation** = one meaningful step inside that execution, such as retrieval, reranking, a tool call, or a generation.


### 🔎 Inspect the answer — not just whether it sounds good

Ask yourself:

- Does it tell you **which document** supports the number?
- Can you inspect the **page or source**?
- If it gives a precise percentage, do you know whether it came from evidence or model memory?
- If the answer is wrong, can you tell *why* it was wrong?

**Key idea:** today we will make the evidence path increasingly inspectable.


## Concept checkpoint — Temperature: how much freedom does the model have to guess?

### 🧠 Concept

`call_model(messages, temperature=0.3)` in `llm/client.py` passes `temperature` straight through to the chat-completion call. Temperature controls how much randomness gets injected when the model picks its next token:

- **Low temperature (0.0–0.3):** the model prefers its highest-probability tokens → more repeatable, more literal answers. Good when you want the same well-supported answer every time you ask.
- **High temperature (0.7–1.0+):** the model samples more freely → more varied phrasing, but also a higher chance of drifting into an invented number, date, or fact.

For an evidence-first research system we almost always want **low temperature**. We are not looking for creative writing — we are looking for consistent, checkable answers.

Temperature does **not** fix hallucination or knowledge cutoff by itself; it only changes how much the model explores beyond its single most-likely next token.


In [ ]:
# ✍️ YOUR TURN 3 — Compare the same question at two temperatures.
low_temp_answer = call_model(build_messages(question), temperature=0.0)
high_temp_answer = call_model(build_messages(question), temperature=1.0)

print('--- temperature=0.0 ---')
print(low_temp_answer)
print('\n--- temperature=1.0 ---')
print(high_temp_answer)


### 🔎 Inspect — rerun at temperature=1.0 a couple of times

Rerun just the cell above two or three times.

- Did the wording change between runs? Did any numbers change?
- At `temperature=0.0`, is the answer stable across reruns?

✅ **Checkpoint:** For a system that must cite the same evidence the same way every time, which temperature would you default to — and is there ever a good reason to reach for the other one?


## Concept checkpoint — Prompt caching: reuse repeated prefixes, not answers
Long requests often repeat the same **system instructions, tool schemas, and examples**. Many providers can reuse computation for an identical/stable prompt prefix.

```text
stable prefix: system instructions + tool schemas + reusable examples
dynamic suffix: current question + fresh retrieved evidence
```

Prompt caching can reduce repeated input work, latency, or cost depending on the provider. It does **not** update model knowledge, extend context, or guarantee correctness. It is different from caching the final answer.


## Mission 2 — Structured outputs: turn prose into a software interface

### 🧠 Concept

Humans can interpret a paragraph. Software needs predictable fields.

For our research system we want an answer shaped like:

```text
ResearchAnswer
├─ answer
├─ citations[]
│  ├─ source_id
│  └─ page
└─ confidence
```

Structure improves **reliability of the interface**. It does **not** make the facts true by itself.


📎 **Script reference:** `llm/schemas.py` — the `ResearchAnswer`/`Citation` Pydantic models we validate against, plus `parse_structured_answer` for turning raw JSON text into one. Run it standalone with `uv run python -m llm.schemas`.


In [ ]:
from llm.schemas import ResearchAnswer, Citation, parse_structured_answer

ResearchAnswer.model_json_schema()


Now ask the model itself to follow that schema, instead of building an example by hand. `call_model` takes an optional `response_format` — `{'type': 'json_object'}` is plain JSON mode, which far more models (including whatever `openrouter/free` routes you to today) support than the stricter `json_schema` mode.


In [ ]:
structured_prompt = build_messages(question, context='No evidence was supplied for this question.')
structured_prompt[0]['content'] += (
    '\n\nRespond with ONLY a JSON object matching this schema (no prose, no markdown fences):\n'
    + str(ResearchAnswer.model_json_schema())
)

raw = call_model(structured_prompt, response_format={'type': 'json_object'})
print(raw)


In [ ]:
# ✍️ YOUR TURN 4 — Validate the model's own JSON against the schema.
structured_example = None  # TODO: parse_structured_answer(raw)
structured_example


<details><summary><b>Hint</b></summary>
`parse_structured_answer` is already imported from `llm.schemas` — it does `ResearchAnswer.model_validate(json.loads(text))`. If it raises, print `raw` and look at what the model actually returned.
</details>


---
**Next:** open `02_evidence_layer.ipynb` to start building the corpus these answers should actually be grounded in.
